# Automation readiness: small-cap continuation

## tl;dr

**Non ancora pronto per il live.** Sul campione regular-only di 15 trade, a 100 bps il profit factor è 1.15, l'expectancy è 0.39% e la mediana è -1.10%. La metà cronologicamente più recente ha expectancy -1.77%. La pipeline è adatta al prossimo stadio: IBKR shadow/paper testing con ordini non trasmessi.

## Context & Methods

Obiettivo: decidere se il motore può passare dal backtest a un sistema automatico su IBKR o MetaTrader. Il test usa solo sessione regolare, target 20%, dati causali e nessun override di livello.

### Key Assumptions

- Capitale di riferimento: 100.000 USD, rischio 1% e posizione massima 25%.
- I costi sono stress deterministici, non spread bid/ask osservati.
- I risultati sono descrittivi: il campione non è indipendente né out-of-sample.

## Data

### 1. Load cached bars and rerun the battery

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))
from smallcap_bt.config import load_config
from smallcap_bt.data import download_universe, provider_from_config, read_tickers
from smallcap_bt.robustness import run_readiness_analysis

cfg = load_config(ROOT / 'configs/iwm_regular_only.yaml')
tickers = read_tickers(ROOT / 'outputs/iwm_automated/intraday_candidates.txt')
frames, errors = download_universe(tickers, provider_from_config(cfg.data))
assert not errors, errors
results = run_readiness_analysis(frames, cfg, ROOT / 'outputs/automation_readiness')
print({'tickers': len(frames), 'bars': sum(map(len, frames.values()))})

## Results

### 2. Live-readiness gates

In [ ]:
results['readiness_scorecard']

### 3. Cost and latency stress

The bars compare filled-trade expectancy. Extra 15-minute delay is an outage stress, not an estimate of normal broker latency.

In [ ]:
stress = results['execution_stress']
fig, ax = plt.subplots(figsize=(9, 4.5))
for delay, group in stress.groupby('delay_bars'):
    ax.plot(group.total_cost_bps, group.expectancy_filled_pct * 100, marker='o', label=f'{delay} extra bars')
ax.axhline(0, color='#444444', linewidth=1)
ax.set(title='Expectancy under cost and execution-delay stress', xlabel='Total round-trip cost (bps)', ylabel='Mean return per filled trade (%)')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=.2)
plt.show()

### 4. Parameter sensitivity

One parameter is changed at a time. Positive averages alone are insufficient; the production gate also requires profit factor ≥1.30 and a positive median.

In [ ]:
sensitivity = results['parameter_sensitivity'].copy()
sensitivity[['scenario', 'filled_trades', 'profit_factor', 'expectancy_filled_pct', 'median_return_pct']]

### 5. Liquidity and concentration

In [ ]:
liquidity = results['liquidity_diagnostics'].sort_values('participation_of_15m_volume', ascending=False)
liquidity[['ticker', 'shares_at_100k', 'entry_bar_dollar_volume', 'participation_of_15m_volume', 'entry_bar_range_pct']]

### 6. Temporal stability, uncertainty, and session-definition overlap

In [ ]:
display(results['chronological_split'])
display(results['bootstrap_summary'])
display(results['signal_overlap'])

## Takeaways

- **Automation logic:** implementable and reproducible.
- **Economic robustness:** not demonstrated; the recent half is negative and 100-bps costs reduce PF below the live threshold.
- **Execution:** one signal requires an implausibly high share of its 15-minute volume at the reference account size.
- **Next stage:** IBKR shadow mode, followed by at least 50 paper trades; no live order transmission.